In [1]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Widgets: Ticker, RSI period, BB window, Refresh button
ticker_input = widgets.Dropdown(
    options=['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA'],
    value='AAPL',
    description='Ticker:'
)

rsi_slider = widgets.IntSlider(value=14, min=5, max=30, step=1, description='RSI:')
bb_slider = widgets.IntSlider(value=20, min=10, max=50, step=1, description='BB:')
run_button = widgets.Button(description="🔄 Run Analysis", button_style='primary')

display(ticker_input, rsi_slider, bb_slider, run_button)

# Indicator Functions
def calculate_rsi(data, period=14):
    delta = data['Close'].diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    data['RSI'] = rsi
    return data

def calculate_macd(data):
    ema12 = data['Close'].ewm(span=12, adjust=False).mean()
    ema26 = data['Close'].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False).mean()
    data['MACD'] = macd
    data['Signal'] = signal
    return data

def calculate_bollinger_bands(data, window=20):
    sma = data['Close'].rolling(window=window).mean()
    std = data['Close'].rolling(window=window).std()
    data['BB_Upper'] = sma + 2 * std
    data['BB_Lower'] = sma - 2 * std
    return data

def plot_indicators(data, ticker):
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
    ax1.plot(data['Close'], label='Close', color='black')
    ax1.plot(data['BB_Upper'], '--', label='BB Upper', color='gray')
    ax1.plot(data['BB_Lower'], '--', label='BB Lower', color='gray')
    ax1.set_title(f"{ticker} - Price + Bollinger Bands")
    ax1.legend()

    ax2.plot(data['RSI'], label='RSI', color='purple')
    ax2.axhline(70, color='red', linestyle='--')
    ax2.axhline(30, color='green', linestyle='--')
    ax2.set_title('RSI')
    ax2.legend()

    ax3.plot(data['MACD'], label='MACD', color='blue')
    ax3.plot(data['Signal'], label='Signal', color='orange')
    ax3.set_title('MACD')
    ax3.legend()

    plt.tight_layout()
    plt.show()

# Callback function for the button
def run_analysis(_):
    clear_output(wait=True)
    display(ticker_input, rsi_slider, bb_slider, run_button)
    
    ticker = ticker_input.value
    df = yf.download(ticker, start="2024-01-01", end="2025-01-01")
    df = calculate_rsi(df, rsi_slider.value)
    df = calculate_macd(df)
    df = calculate_bollinger_bands(df, bb_slider.value)
    plot_indicators(df, ticker)

run_button.on_click(run_analysis)


Dropdown(description='Ticker:', options=('AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA'), value='AAPL')

IntSlider(value=14, description='RSI:', max=30, min=5)

IntSlider(value=20, description='BB:', max=50, min=10)

Button(button_style='primary', description='🔄 Run Analysis', style=ButtonStyle())

In [2]:
import yfinance as yf
yf.download("AAPL", start="2024-01-01", end="2025-01-01")


C:\Users\johnl\AppData\Local\Temp\ipykernel_20520\1619053232.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  yf.download("AAPL", start="2024-01-01", end="2025-01-01")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2024-01-02,184.290405,187.070052,182.553128,185.789422,82488700
2024-01-03,182.910507,184.528662,182.096461,182.880727,58414500
2024-01-04,180.587524,181.758939,179.565014,180.825770,71983600
2024-01-05,179.862854,181.431370,178.860202,180.666978,62303300
2024-01-08,184.210983,184.250701,180.180502,180.766209,59144500
...,...,...,...,...,...
2024-12-24,257.578674,257.588630,254.675658,254.875189,23234700
2024-12-26,258.396667,259.474086,257.010028,257.568678,27237100


In [3]:
!pip install --upgrade yfinance


In [4]:
import yfinance as yf
data = yf.download("AAPL", start="2024-01-01", end="2025-01-01")
print(data.head())


C:\Users\johnl\AppData\Local\Temp\ipykernel_20520\2382597126.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("AAPL", start="2024-01-01", end="2025-01-01")
[*********************100%***********************]  1 of 1 completed

Price            Close        High         Low        Open    Volume
Ticker            AAPL        AAPL        AAPL        AAPL      AAPL
Date                                                                
2024-01-02  184.290405  187.070052  182.553128  185.789422  82488700
2024-01-03  182.910507  184.528662  182.096461  182.880727  58414500
2024-01-04  180.587524  181.758939  179.565014  180.825770  71983600
2024-01-05  179.862854  181.431370  178.860202  180.666978  62303300
2024-01-08  184.210983  184.250701  180.180502  180.766209  59144500


In [5]:
# Add to your imports if not already there
from datetime import date
import os

# Add date selectors
start_date = widgets.DatePicker(
    description='Start Date:',
    value=date(2024, 1, 1)
)
end_date = widgets.DatePicker(
    description='End Date:',
    value=date(2025, 1, 1)
)

# Add export toggle
export_checkbox = widgets.Checkbox(
    value=False,
    description='💾 Save to CSV'
)

# Show all widgets
display(ticker_input, rsi_slider, bb_slider, start_date, end_date, export_checkbox, run_button)

# Update the callback function
def run_analysis(_):
    clear_output(wait=True)
    display(ticker_input, rsi_slider, bb_slider, start_date, end_date, export_checkbox, run_button)

    ticker = ticker_input.value
    start = start_date.value.strftime('%Y-%m-%d')
    end = end_date.value.strftime('%Y-%m-%d')

    try:
        df = yf.download(ticker, start=start, end=end)
        df = calculate_rsi(df, rsi_slider.value)
        df = calculate_macd(df)
        df = calculate_bollinger_bands(df, bb_slider.value)
        plot_indicators(df, ticker)

        if export_checkbox.value:
            filename = f"{ticker}_{start}_to_{end}_indicators.csv"
            df.to_csv(filename)
            print(f"✅ Data saved to '{filename}' in: {os.getcwd()}")

    except Exception as e:
        print(f"⚠️ Error: {e}")


Dropdown(description='Ticker:', options=('AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA'), value='AAPL')

IntSlider(value=14, description='RSI:', max=30, min=5)

IntSlider(value=20, description='BB:', max=50, min=10)

DatePicker(value=datetime.date(2024, 1, 1), description='Start Date:', step=1)

DatePicker(value=datetime.date(2025, 1, 1), description='End Date:', step=1)

Checkbox(value=False, description='💾 Save to CSV')

Button(button_style='primary', description='🔄 Run Analysis', style=ButtonStyle())

In [6]:
nvda


NameError: name 'nvda' is not defined

In [ ]:
pip install ta-lib


In [ ]:
conda install -c conda-forge ta-lib


In [ ]:
conda install -c conda-forge ta-lib


In [ ]:
import talib

# Example: Detect hammer candlesticks
def detect_candlestick_patterns(data):
    patterns = {
        'Hammer': talib.CDLHAMMER(data['Open'], data['High'], data['Low'], data['Close']),
        'Engulfing': talib.CDLENGULFING(data['Open'], data['High'], data['Low'], data['Close']),
        'Doji': talib.CDLDOJI(data['Open'], data['High'], data['Low'], data['Close']),
    }
    for name, values in patterns.items():
        data[name] = values
    return data



In [ ]:
pip install ta-lib


In [ ]:
pip install ta-lib


In [7]:
pip install ta-lib


  Using cached ta_lib-0.6.4.tar.gz (381 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Failed to build ta-lib
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  exit code: 1
  
  [34 lines of output]
  <string>:83: UserWarning: Cannot find ta-lib library, installation may fail.
  C:\Users\johnl\AppData\Local\Temp\pip-build-env-s9fy4hht\overlay\Lib\site-packages\setuptools\config\_apply_pyprojecttoml.py:82: SetuptoolsWarning: `install_requires` overwritten in `pyproject.toml` (dependencies)
    corresp(dist, value, root_dir)
  running bdist_wheel
  running build
  running build_py
  creating build\lib.win-amd64-cpython-313\talib
  copying talib\abstract.py -> build\lib.win-amd64-cpython-313\talib
  copying talib\deprecated.py -> build\lib.win-amd64-cpython-313\talib
  copying talib\stream.py -> build\lib.win-amd64-cpython-313\talib
  copying talib\__init__.py -> build\lib.win-amd64-cpython-313\talib
  running egg_info
  writing ta_lib.egg-info\PKG-INFO
  writing dependency_links to ta_lib.egg-info\dependency_links.txt
  writing requirements to ta_lib.egg-info\requires.txt
  writing top-level names to t

In [8]:
conda install -c conda-forge ta-lib


Jupyter detected...
Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: failed

Note: you may need to restart the kernel to use updated packages.



LibMambaUnsatisfiableError: Encountered problems while solving:
  - package ta-lib-0.4.19-py310h9b08ddd_4 requires python >=3.10,<3.11.0a0, but none of the providers can be installed

Could not solve for environment specs
The following packages are incompatible
\u251c\u2500 pin on python 3.13.* =* * is installable and it requires
\u2502  \u2514\u2500 python =3.13 *, which can be installed;
\u2514\u2500 ta-lib =* * is not installable because there are no viable options
   \u251c\u2500 ta-lib [0.4.19|0.4.31|0.4.32|0.5.1] would require
   \u2502  \u2514\u2500 python >=3.10,<3.11.0a0 *, which conflicts with any installable versions previously reported;
   \u251c\u2500 ta-lib [0.4.18|0.4.19|0.4.31|0.4.32] would require
   \u2502  \u2514\u2500 python >=3.8,<3.9.0a0 *, which conflicts with any installable versions previously reported;
   \u251c\u2500 ta-lib [0.4.19|0.4.31|0.4.32|0.5.1] would require
   \u2502  \u2514\u2500 python >=3.9,<3.10.0a0 *, which conflicts with any installable versio

In [9]:
conda install -c conda-forge ta-lib


Jupyter detected...
Note: you may need to restart the kernel to use updated packages.

Channels:
 - conda-forge
 - defaults
Platform: win-64
Solving environment: failed



LibMambaUnsatisfiableError: Encountered problems while solving:
  - package ta-lib-0.4.19-py310h9b08ddd_4 requires python >=3.10,<3.11.0a0, but none of the providers can be installed

Could not solve for environment specs
The following packages are incompatible
\u251c\u2500 pin on python 3.13.* =* * is installable and it requires
\u2502  \u2514\u2500 python =3.13 *, which can be installed;
\u2514\u2500 ta-lib =* * is not installable because there are no viable options
   \u251c\u2500 ta-lib [0.4.19|0.4.31|0.4.32|0.5.1] would require
   \u2502  \u2514\u2500 python >=3.10,<3.11.0a0 *, which conflicts with any installable versions previously reported;
   \u251c\u2500 ta-lib [0.4.18|0.4.19|0.4.31|0.4.32] would require
   \u2502  \u2514\u2500 python >=3.8,<3.9.0a0 *, which conflicts with any installable versions previously reported;
   \u251c\u2500 ta-lib [0.4.19|0.4.31|0.4.32|0.5.1] would require
   \u2502  \u2514\u2500 python >=3.9,<3.10.0a0 *, which conflicts with any installable versio

In [10]:
import talib

# Example: Detect hammer candlesticks
def detect_candlestick_patterns(data):
    patterns = {
        'Hammer': talib.CDLHAMMER(data['Open'], data['High'], data['Low'], data['Close']),
        'Engulfing': talib.CDLENGULFING(data['Open'], data['High'], data['Low'], data['Close']),
        'Doji': talib.CDLDOJI(data['Open'], data['High'], data['Low'], data['Close']),
    }
    for name, values in patterns.items():
        data[name] = values
    return data


ModuleNotFoundError: No module named 'talib'

In [ ]:
df = detect_candlestick_patterns(df)


In [ ]:
for pattern in ['Hammer', 'Engulfing', 'Doji']:
    matches = df[df[pattern] != 0]
    if not matches.empty:
        print(f"\n📌 {pattern} detected on:")
        print(matches[[pattern]].tail(5))


In [ ]:
def plot_indicators(data, ticker):
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

    ax1.plot(data['Close'], label='Close', color='black')
    ax1.plot(data['BB_Upper'], '--', label='BB Upper', color='gray')
    ax1.plot(data['BB_Lower'], '--', label='BB Lower', color='gray')

    # ➕ Highlight patterns
    for pattern, marker, color in [
        ('Hammer', '^', 'green'),
        ('Engulfing', 's', 'orange'),
        ('Doji', 'X', 'red')
    ]:
        matches = data[data[pattern] != 0]
        ax1.scatter(matches.index, matches['Close'], marker=marker, color=color, label=pattern)

    ax1.set_title(f"{ticker} - Price + Bollinger Bands + Patterns")
    ax1.legend()

    ax2.plot(data['RSI'], label='RSI', color='purple')
    ax2.axhline(70, color='red', linestyle='--')
    ax2.axhline(30, color='green', linestyle='--')
    ax2.set_title('RSI')
    ax2.legend()

    ax3.plot(data['MACD'], label='MACD', color='blue')
    ax3.plot(data['Signal'], label='Signal', color='orange')
    ax3.set_title('MACD')
    ax3.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def apply_kmeans_clustering(data, features=['RSI', 'MACD', 'BB_Upper', 'BB_Lower'], n_clusters=3):
    df_cluster = data.dropna().copy()
    X = df_cluster[features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=n_clusters, n_init='auto', random_state=42)
    df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)

    # Merge back with original DataFrame
    data = data.merge(df_cluster['Cluster'], left_index=True, right_index=True, how='left')
    return data


In [ ]:
df = apply_kmeans_clustering(df)


In [ ]:
for cluster in sorted(df['Cluster'].dropna().unique()):
    cluster_data = df[df['Cluster'] == cluster]
    ax1.plot(cluster_data.index, cluster_data['Close'], label=f'Regime {cluster}')


In [ ]:
def generate_signals(data):
    signals = []

    for i in range(1, len(data)):
        if data['RSI'].iloc[i - 1] > 30 and data['RSI'].iloc[i] <= 30:
            signals.append('RSI Buy')
        elif data['RSI'].iloc[i - 1] < 70 and data['RSI'].iloc[i] >= 70:
            signals.append('RSI Sell')
        elif data['MACD'].iloc[i - 1] < data['Signal'].iloc[i - 1] and data['MACD'].iloc[i] > data['Signal'].iloc[i]:
            signals.append('MACD Buy')
        elif data['MACD'].iloc[i - 1] > data['Signal'].iloc[i - 1] and data['MACD'].iloc[i] < data['Signal'].iloc[i]:
            signals.append('MACD Sell')
        else:
            signals.append('')
    
    signals.insert(0, '')  # First row has no prior data
    data['Signal_Alert'] = signals
    return data


In [ ]:
df = generate_signals(df)


In [ ]:
signal_hits = df[df['Signal_Alert'] != '']
print("📍 Trade Signals:")
print(signal_hits[['Close', 'Signal_Alert']].tail(10))
